[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tnecniv-Teikram/NSA-hyperspectral-course-2026/blob/main/exercises/ex_05_blue_carbon.ipynb)

*Running in Colab? Run the setup cell below first — it clones the course and installs what this notebook needs. Running locally? It does nothing.*

# 🌿 Exercises: Ecosystem health, biodiversity & blue carbon
### Space Academy 2026 · Hyperspectral course · Week 2 — Project scoping and use case definition (Practical)

Check how good the simple mangrove proxy of notebook 05 really is, find out what makes mangroves spectrally unique, learn a better rule from data, and turn the result into a blue-carbon estimate — with honest uncertainty — for the mangroves of Jiquilisco Bay, El Salvador.

These exercises accompany the theme notebook [`05_ecosystem_health_blue_carbon.ipynb`](../../notebooks/05_ecosystem_health_blue_carbon.ipynb). Run that notebook first
(or at least read it) — here we reuse its scene and indices, loaded with the course helper module `hsi_utils`.

> Work through the exercises in order — they get harder (⭐ → ⭐⭐⭐). Each has a hint; worked solutions are in `exercises/solutions/`.

**What you will practise**
- Precision, recall and IoU of a map against a reference
- Spectral differences between mangroves and other vegetation (NIR, SWIR, red edge)
- Training and spatially testing an interpretable decision tree
- Why a model only knows the classes it was trained on
- Propagating map and carbon-density uncertainty into a carbon stock

**Data**
- `20250223_165546_32_4001` — Jiquilisco Bay, Usulután, El Salvador (23 Feb 2025)
- `data/labels/…_worldcover2021_30m.tif` — ESA WorldCover 2021 on the Tanager grid (class 95 = mangroves)

In [ ]:
# --- Setup: works both in Google Colab and in a local environment --------------------
# In Colab this clones the course repository, installs the geospatial packages that Colab
# does not ship, and moves into the course folder so the notebook finds hsi_utils and data/.
# Locally it does nothing at all — your own environment is already active.
import os
import pathlib
import subprocess
import sys

REPO = "Tnecniv-Teikram/NSA-hyperspectral-course-2026"          # change with: python tools/set_repo_url.py <owner>/<repo>

if "google.colab" in sys.modules:
    course = pathlib.Path("/content/hyperspectral-course")
    if not course.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q",
                        f"https://github.com/{REPO}.git", str(course)], check=True)
    os.chdir(course)
    sys.path.insert(0, str(course))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "h5py", "rasterio", "rioxarray", "xarray", "geopandas"], check=True)
    print("Colab ready — course at", course)
else:
    print("Local environment:", sys.executable.split("/")[-3])


## Setup

In [ ]:
# hsi_utils.py lives in the course folder (.. from here)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# On Google Colab: upload hsi_utils.py next to this notebook, change the line above to
# sys.path.insert(0, ".") and uncomment:
# %pip install -q h5py numpy scipy matplotlib scikit-learn pandas rasterio requests

import numpy as np
import matplotlib.pyplot as plt
import hsi_utils as hu

plt.rcParams["figure.dpi"] = 90

Load the scene and the bands used in the theme notebook. `hu.get_scene()` finds the local HDF5 or downloads it (~1 GB) from Planet's STAC catalogue.

In [ ]:
import pandas as pd

SCENE_ID = "20250223_165546_32_4001"   # Jiquilisco Bay, Usulután, El Salvador (coastal-water-bodies)
scene = hu.get_scene(SCENE_ID)
print(scene.summary())

valid = scene.valid_mask()
B = {w: scene.read_band(w) for w in (443, 560, 665, 705, 708, 800, 860, 1650, 2200)}
NDVI = hu.nd(B[800], B[665])            # as in notebook 05
NDWI = hu.nd(B[560], B[860])
NDCI = hu.nd(B[708], B[665])
mangrove_proxy = valid & (NDVI > 0.2) & (NDWI < 0)   # the mask behind notebook 05's "Mangrove proxy"
PIXEL_HA = 0.09

wc, purity = hu.load_labels(SCENE_ID)     # ESA WorldCover 2021 on the Tanager grid (95 = mangroves)
wc_mangrove = valid & (wc == 95)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].imshow(scene.rgb()); axes[0].set_title("True colour")
axes[1].imshow(np.where(mangrove_proxy, 1, np.nan), cmap="Greens", vmin=0, vmax=1.3); axes[1].set_title("Notebook 05 proxy: NDVI > 0.2 & NDWI < 0")
axes[2].imshow(np.where(wc_mangrove, 1, np.nan), cmap="Greens", vmin=0, vmax=1.3); axes[2].set_title("WorldCover 2021: mangroves")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## Exercise 1 — Proxy area vs reference area  ⭐

1. Compute the area (ha) of the notebook 05 mangrove proxy and of the WorldCover mangrove class.
2. Compute the agreement statistics of the proxy against WorldCover: precision (what fraction of the proxy is
   WorldCover mangrove), recall (what fraction of WorldCover mangrove the proxy finds) and IoU
   (intersection over union).
3. What does the proxy actually map?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

Precision = |proxy ∩ ref| / |proxy|, recall = |proxy ∩ ref| / |ref|, IoU = |proxy ∩ ref| / |proxy ∪ ref|,
where |·| is a pixel count (`.sum()` of a boolean array).

</details>

## Exercise 2 — What makes a mangrove spectrally different?  ⭐⭐

Compare pure WorldCover **mangrove** (95) pixels with pure **tree cover** (10) and **cropland** (40):

1. Plot the mean spectra of the three classes.
2. Plot their first derivative (dR/dλ) between 650 and 800 nm and report the wavelength of the maximum slope
   (the red-edge position) for each class.
3. In which spectral regions are mangroves most different from the other vegetation?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

Sample up to 500 pixels per class (as in the previous notebooks) and average. The derivative:
`np.gradient(spectrum, wl)`. Restrict to 650–800 nm before `np.argmax`.

</details>

## Exercise 3 — A better mangrove rule, learned from data  ⭐⭐⭐

Train a small, interpretable **decision tree** that separates mangroves from other vegetation, and test it
**on a different part of the scene**:

1. Features per pixel: NDVI, NDWI, NDCI, NDRE (860/705), NDMI (860/1650) and reflectance at 860 and 1650 nm.
2. Training data: pure WorldCover pixels (purity = 9) of classes 95 (mangrove = 1) and 10, 30, 40 (other
   vegetation = 0) in the **western** part of the scene (column < 390). Test data: the same classes in the
   **eastern** part (column ≥ 390).
3. Fit `DecisionTreeClassifier(max_depth=2, class_weight="balanced")`, print the rules and report precision,
   recall and IoU on the test pixels. Compare with the notebook 05 proxy on the same test pixels.
4. Apply the tree to **all** valid pixels of the eastern half. What goes wrong if you don't first restrict it to
   vegetated pixels?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

Stack the features into an array of shape (rows, cols, n_features) with `np.stack([...], axis=-1)`;
`X[mask]` then gives an (n_pixels, n_features) matrix. `sklearn.tree.export_text(clf, feature_names=names)` prints
the rules. Column index of every pixel: `cols = np.indices(wc.shape)[1]`.

</details>

## Exercise 4 — From hectares to blue carbon — with uncertainty  ⭐⭐

Blue-carbon projects convert **mapped area × carbon density** into a carbon stock. Published
total-ecosystem carbon densities for mangroves (trees + soil down to ~1 m) vary a lot between regions and
studies. For this exercise use a *low* (400), *central* (700) and *high* (1000) value in Mg C per hectare,
as a plausible range (see e.g. Donato et al., 2011; Kauffman et al., 2020 for real regional values).

1. Compute the stock (Mg C and Mt CO₂-equivalent; 1 Mg C = 44/12 Mg CO₂) for three area estimates: the proxy,
   WorldCover, and your decision tree (inside the vegetation mask, whole scene).
2. Which uncertainty is larger here — the choice of map or the choice of carbon density?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

To apply the tree to the whole scene: `pred = np.zeros(valid.shape, bool); pred[m] = clf.predict(X[m])` with
`m = valid & finite`, then `& mangrove_proxy`.

</details>

## Exercise 5 — Reflection — what would it take to trust a mangrove map?  ⭐

In a short markdown cell: WorldCover served as our "truth", but it is itself a model output from 2021 with a
10 m grid. Describe how you would **validate** a Tanager mangrove map for a blue-carbon project. Cover the
reference data source, the sampling design and the reported metrics.

In [ ]:
# optional scratch space — answer in a markdown cell

<details><summary>💡 Hint</summary>

Think of very-high-resolution imagery (Planet VHR, drones), field plots, Global Mangrove Watch, stratified
random sampling (Olofsson et al., 2014), area-adjusted accuracy and confidence intervals.

</details>

## Key takeaways

- 'NDVI > 0.2 and not water' maps vegetation, not mangroves — check proxies against a reference.
- Mangroves are darkest in the SWIR; hyperspectral SWIR bands separate them from inland forest.
- Test models on a different area (spatial hold-out) and apply them only where their training classes exist.
- Carbon stocks need both a validated area and regional carbon densities — report a range.

---

> 📖 Data: Tanager — CC-BY-4.0 © Planet Labs PBC  
> 📖 Labels: ESA WorldCover 2021 v200 — CC-BY 4.0